# FIFA World Cup Predictor

## Version 10 - The 2026 World Cup tournament structure

### Goal

Version 9 gave us a reusable `predict_world_cup_match(team_a, team_b, neutral=True)` function.
Before we can simulate the whole tournament, we need a **clean description of the tournament
itself**: which teams, which groups, which venues, and how the knockout bracket is shaped.

This notebook builds that structure as **reusable Python data** (dictionaries, lists and
DataFrames) that the future simulator will consume.

**We do not run any Monte Carlo simulation yet.**

### Source of truth

Everything here uses the **official 2026 FIFA World Cup** facts (the December 5, 2025 final draw,
the 16 official stadiums, and the confirmed 48-team / 12-group / 104-match format). Where an exact
value is not available to us in the data we keep, we **clearly mark it as pending** instead of
inventing it.

# Why keep tournament structure separate from the ML model?

The ML model (Versions 7-9) answers one narrow question: *given two teams and a venue, what are
the outcome probabilities?* It does not know anything about groups, fixtures or brackets.

The tournament structure answers a completely different question: *which matches are played,
in what order, and who advances?* Keeping them separate means:

- We can change the list of fixtures without retraining the model.
- We can reuse the same model for any tournament format.
- The simulator's job becomes simple: read the structure, call the model for each fixture,
  sample an outcome, and update who advances.

Separating *"the match is played at SoFi on June 12"* (structure) from *"USA vs Paraguay is a
58% / 22% / 20% match"* (model) is good engineering and much easier to debug.

# How the 48-team / 12-group format works

The 2026 World Cup is the first with 48 teams (up from 32):

- **12 groups of 4** (Groups A to L), 104 matches total, run from 11 June to 19 July 2026.
- Each team plays **3 group matches** (a round-robin; 6 matches per group, 72 group matches).
- **Group stage** outcomes are: Team A wins / Draw / Team B wins. A draw is a valid result and
  both teams get a point.
- After the group stage, **32 teams advance**: the top 2 of each group (24) plus the **8 best
  third-placed teams**.
- Those 32 enter the **Round of 32**, then Round of 16, Quarter-finals, Semi-finals and the Final
  (plus a third-place match).

### Group stage vs knockout - the key difference

- **Group stage:** three possible outcomes; a draw is fine; teams are ranked by points.
- **Knockout:** a winner MUST be found; level after 90 minutes continues to extra time then
  penalties; only one team advances.

Our three-outcome probabilities fit the group stage directly. Knockouts need the extra-time /
penalties handling we introduced as an approximation in Version 9.

In [1]:
import pandas as pd
import numpy as np

# Section 1 - The 16 official venues

FIFA selected 16 stadiums in three host countries: 11 in the United States, 3 in Mexico and
2 in Canada. We store the FIFA tournament name, the real stadium name, the city and country.

In [2]:
VENUES = [
    {'fifa_name': 'Atlanta Stadium',             'stadium': 'Mercedes-Benz Stadium', 'city': 'Atlanta',      'country': 'United States'},
    {'fifa_name': 'Boston Stadium',              'stadium': 'Gillette Stadium',      'city': 'Boston',       'country': 'United States'},
    {'fifa_name': 'Dallas Stadium',              'stadium': 'AT&T Stadium',          'city': 'Dallas',       'country': 'United States'},
    {'fifa_name': 'Houston Stadium',             'stadium': 'NRG Stadium',           'city': 'Houston',      'country': 'United States'},
    {'fifa_name': 'Kansas City Stadium',         'stadium': 'Arrowhead Stadium',     'city': 'Kansas City',  'country': 'United States'},
    {'fifa_name': 'Los Angeles Stadium',         'stadium': 'SoFi Stadium',          'city': 'Los Angeles',  'country': 'United States'},
    {'fifa_name': 'Miami Stadium',               'stadium': 'Hard Rock Stadium',     'city': 'Miami',        'country': 'United States'},
    {'fifa_name': 'New York New Jersey Stadium', 'stadium': 'MetLife Stadium',       'city': 'New York',     'country': 'United States'},
    {'fifa_name': 'Philadelphia Stadium',        'stadium': 'Lincoln Financial Field','city': 'Philadelphia','country': 'United States'},
    {'fifa_name': 'San Francisco Bay Area Stadium','stadium': "Levi's Stadium",     'city': 'San Francisco','country': 'United States'},
    {'fifa_name': 'Seattle Stadium',             'stadium': 'Lumen Field',           'city': 'Seattle',      'country': 'United States'},
    {'fifa_name': 'Toronto Stadium',             'stadium': 'BMO Field',             'city': 'Toronto',      'country': 'Canada'},
    {'fifa_name': 'BC Place Vancouver',          'stadium': 'BC Place',              'city': 'Vancouver',    'country': 'Canada'},
    {'fifa_name': 'Mexico City Stadium',         'stadium': 'Estadio Azteca',        'city': 'Mexico City',  'country': 'Mexico'},
    {'fifa_name': 'Guadalajara Stadium',         'stadium': 'Estadio Akron',         'city': 'Guadalajara',  'country': 'Mexico'},
    {'fifa_name': 'Monterrey Stadium',           'stadium': 'Estadio BBVA',          'city': 'Monterrey',    'country': 'Mexico'},
]

venues_df = pd.DataFrame(VENUES)
print("Number of venues:", len(VENUES))
print(venues_df['country'].value_counts())
venues_df

Number of venues: 16
country
United States    11
Mexico            3
Canada            2
Name: count, dtype: int64


,fifa_name,stadium,city,country
0,Atlanta Stadium,Mercedes-Benz Stadium,Atlanta,United States
1,Boston Stadium,Gillette Stadium,Boston,United States
2,Dallas Stadium,AT&T Stadium,Dallas,United States
3,Houston Stadium,NRG Stadium,Houston,United States
4,Kansas City Stadium,Arrowhead Stadium,Kansas City,United States
5,Los Angeles Stadium,SoFi Stadium,Los Angeles,United States
6,Miami Stadium,Hard Rock Stadium,Miami,United States
7,New York New Jersey Stadium,MetLife Stadium,New York,United States
8,Philadelphia Stadium,Lincoln Financial Field,Philadelphia,United States
9,San Francisco Bay Area Stadium,Levi's Stadium,San Francisco,United States


# Section 2 - The 48 qualified teams and the 12 groups

The final draw (5 December 2025) produced these groups. The three co-hosts - USA, Canada and
Mexico - qualified automatically. Tournament debutants include Curaçao and Cape Verde.

In [3]:
GROUPS = {
    'A': ['Mexico', 'South Africa', 'South Korea', 'Czechia'],
    'B': ['Canada', 'Bosnia and Herzegovina', 'Qatar', 'Switzerland'],
    'C': ['Brazil', 'Morocco', 'Haiti', 'Scotland'],
    'D': ['United States', 'Paraguay', 'Australia', 'Türkiye'],
    'E': ['Germany', 'Curaçao', 'Ivory Coast', 'Ecuador'],
    'F': ['Netherlands', 'Japan', 'Sweden', 'Tunisia'],
    'G': ['Belgium', 'Egypt', 'Iran', 'New Zealand'],
    'H': ['Spain', 'Cape Verde', 'Saudi Arabia', 'Uruguay'],
    'I': ['France', 'Senegal', 'Iraq', 'Norway'],
    'J': ['Argentina', 'Algeria', 'Austria', 'Jordan'],
    'K': ['Portugal', 'DR Congo', 'Uzbekistan', 'Colombia'],
    'L': ['England', 'Croatia', 'Ghana', 'Panama'],
}

all_teams = sorted({t for team_list in GROUPS.values() for t in team_list})
print("Number of qualified teams:", len(all_teams))
print("Number of groups:", len(GROUPS))
print()
pd.DataFrame({"Group": list(GROUPS.keys()), "Teams": [", ".join(v) for v in GROUPS.values()]})

Number of qualified teams: 48
Number of groups: 12



,Group,Teams
0,A,"Mexico, South Africa, South Korea, Czechia"
1,B,"Canada, Bosnia and Herzegovina, Qatar, Switzer..."
2,C,"Brazil, Morocco, Haiti, Scotland"
3,D,"United States, Paraguay, Australia, Türkiye"
4,E,"Germany, Curaçao, Ivory Coast, Ecuador"
5,F,"Netherlands, Japan, Sweden, Tunisia"
6,G,"Belgium, Egypt, Iran, New Zealand"
7,H,"Spain, Cape Verde, Saudi Arabia, Uruguay"
8,I,"France, Senegal, Iraq, Norway"
9,J,"Argentina, Algeria, Austria, Jordan"


# Section 3 - Match our team names to the model's team names

Version 9's predictor reads team names from the historical dataset. Some official 2026 names differ
from the dataset, so we map them to the exact dataset spelling (the same idea as Version 9's
`TEAM_ALIASES`, e.g. `USA -> United States`):

- `Czechia` -> `Czech Republic`
- `Türkiye` / `Turkiye` -> `Turkey`
- `United States` stays `United States`
- `Curaçao` stays `Curaçao` - a debutant with **no match record** in the dataset (Version 9 would
  fall back to its default Elo/form, which we must remember when simulating).

We also list which of our 48 teams the model has historical data for.

In [4]:
# Same idea as Version 9's TEAM_ALIASES, extended for the official 2026 spellings.
TEAM_ALIASES = {
    'USA': 'United States',
    'US': 'United States',
    'Czechia': 'Czech Republic',
    'Türkiye': 'Turkey',
    'Turkiye': 'Turkey',
}

def dataset_name(team):
    return TEAM_ALIASES.get(team, team)

# Which dataset team names actually exist in the historical results.
df_check = pd.read_csv("../data/raw/results.csv")
known_names = set(df_check['home_team']) | set(df_check['away_team'])

rows = []
for team in all_teams:
    ds = dataset_name(team)
    rows.append({
        'Official name': team,
        'Dataset name': ds,
        'Model has data?': ds in known_names,
    })
pd.DataFrame(rows)

,Official name,Dataset name,Model has data?
0,Algeria,Algeria,True
1,Argentina,Argentina,True
2,Australia,Australia,True
3,Austria,Austria,True
4,Belgium,Belgium,True
5,Bosnia and Herzegovina,Bosnia and Herzegovina,True
6,Brazil,Brazil,True
7,Canada,Canada,True
8,Cape Verde,Cape Verde,True
9,Colombia,Colombia,True


# Section 4 - The group-stage fixtures

Every group is a round-robin: each of its 4 teams plays the other 3 (6 matches per group,
72 group matches). We generate those fixtures from the group rosters using a standard 4-team
pairing pattern.

### Venue and host logic

- A World Cup match is **neutral** (`neutral = True`) unless a host nation plays **on its own
  soil**.
- A host only gets home advantage when the fixture is actually played in that host's home
  country (e.g. Mexico at Estadio Azteca, USA at a US venue, Canada in Toronto).
- We confirmed the **three host opening fixtures** from the official schedule (Mexico at
  Mexico City on 11 June; USA at Los Angeles on 12 June; Canada at Toronto on 12 June) and set
  those `neutral = False` with their host nation.
- FIFA's full per-match venue/date list for the remaining group fixtures must be ingested from
  the official schedule; we mark those venues/dates as `TBD (official schedule)` and keep them
  **neutral** until a host match is confirmed. We do **not** invent venues.

Each fixture stores: `match_id`, `group`, `team_a`, `team_b`, `date`, `venue`, `city`, `neutral`,
and `host_nation_if_applicable`.

In [5]:
# Standard 4-team round-robin already implies each team plays exactly 3 matches.
PAIRINGS = [(0, 1), (2, 3), (0, 2), (3, 1), (1, 2), (0, 3)]

VENUE_COUNTRY = {v['fifa_name']: v['country'] for v in VENUES}
HOST_HOME_COUNTRY = {'Mexico': 'Mexico', 'United States': 'United States', 'Canada': 'Canada'}

# Confirmed host opening fixtures -> (venue, city, date)
HOST_OPENERS = {
    ('A', 0): ('Mexico City Stadium', 'Mexico City', '2026-06-11'),  # Mexico opener
    ('D', 0): ('Los Angeles Stadium', 'Los Angeles', '2026-06-12'),  # USA opener
    ('B', 0): ('Toronto Stadium', 'Toronto', '2026-06-12'),          # Canada opener
}

def build_group_fixtures():
    fixtures = []
    match_id = 1
    for group, teams in GROUPS.items():
        for i, (ia, ib) in enumerate(PAIRINGS):
            team_a = teams[ia]
            team_b = teams[ib]

            key = (group, i)
            if key in HOST_OPENERS:
                venue, city, date = HOST_OPENERS[key]
            else:
                venue, city, date = 'TBD (official schedule)', 'TBD', 'TBD'

            # Determine host/neutral from the venue.
            if venue.startswith('TBD'):
                neutral, host = True, None
            else:
                venue_country = VENUE_COUNTRY[venue]
                host = next(
                    (t for t in (team_a, team_b)
                     if HOST_HOME_COUNTRY.get(dataset_name(t)) == venue_country),
                    None,
                )
                neutral = host is None

            fixtures.append({
                'match_id': f"G-{group}-{i + 1}",
                'group': group,
                'team_a': team_a,
                'team_b': team_b,
                'date': date,
                'venue': venue,
                'city': city,
                'neutral': neutral,
                'host_nation_if_applicable': host,
            })
    return fixtures

fixtures = build_group_fixtures()
fixtures_df = pd.DataFrame(fixtures)
print("Total group-stage fixtures:", len(fixtures))
print("Group-stage fixtures are all group round-robin matches (72).")
fixtures_df.head(12)

Total group-stage fixtures: 72
Group-stage fixtures are all group round-robin matches (72).


,match_id,group,team_a,team_b,date,venue,city,neutral,host_nation_if_applicable
0,G-A-1,A,Mexico,South Africa,2026-06-11,Mexico City Stadium,Mexico City,False,Mexico
1,G-A-2,A,South Korea,Czechia,TBD,TBD (official schedule),TBD,True,NaN
2,G-A-3,A,Mexico,South Korea,TBD,TBD (official schedule),TBD,True,NaN
3,G-A-4,A,Czechia,South Africa,TBD,TBD (official schedule),TBD,True,NaN
4,G-A-5,A,South Africa,South Korea,TBD,TBD (official schedule),TBD,True,NaN
5,G-A-6,A,Mexico,Czechia,TBD,TBD (official schedule),TBD,True,NaN
6,G-B-1,B,Canada,Bosnia and Herzegovina,2026-06-12,Toronto Stadium,Toronto,False,Canada
7,G-B-2,B,Qatar,Switzerland,TBD,TBD (official schedule),TBD,True,NaN
8,G-B-3,B,Canada,Qatar,TBD,TBD (official schedule),TBD,True,NaN
9,G-B-4,B,Switzerland,Bosnia and Herzegovina,TBD,TBD (official schedule),TBD,True,NaN


### Host / neutral flags for every fixture

Only the confirmed host-opening fixtures are flagged as host home matches right now. The rest are
neutral until their venue is filled in from the official schedule.

In [6]:
host_matches = fixtures_df[fixtures_df['host_nation_if_applicable'].notna()]
print("Host home-soil matches flagged:", len(host_matches))
host_matches[['match_id', 'group', 'team_a', 'team_b', 'venue', 'neutral', 'host_nation_if_applicable']]

Host home-soil matches flagged: 3


,match_id,group,team_a,team_b,venue,neutral,host_nation_if_applicable
0,G-A-1,A,Mexico,South Africa,Mexico City Stadium,False,Mexico
6,G-B-1,B,Canada,Bosnia and Herzegovina,Toronto Stadium,False,Canada
18,G-D-1,D,United States,Paraguay,Los Angeles Stadium,False,United States


# Section 5 - Validating the tournament structure

Before the simulator uses this data, we check it automatically:

1. Exactly 48 qualified teams.
2. Exactly 12 groups.
3. 4 teams per group.
4. No duplicate team inside a group (and no team in more than one group).
5. Every fixture references two valid teams from its group.
6. Every fixture has all required fields present.
7. Host / neutral flags are consistent (a host match has a host nation and `neutral == False`;
   a neutral match has no host).

In [7]:
REQUIRED_FIELDS = ['match_id', 'group', 'team_a', 'team_b', 'date', 'venue',
                   'city', 'neutral', 'host_nation_if_applicable']

def run_validations():
    results = []

    # 1. exactly 48 unique teams
    results.append(('exactly 48 teams', len(all_teams) == 48, f"{len(all_teams)} teams"))

    # 2. exactly 12 groups
    results.append(('exactly 12 groups', len(GROUPS) == 12, f"{len(GROUPS)} groups"))

    # 3. 4 teams per group
    sizes = [len(v) for v in GROUPS.values()]
    results.append(('4 teams per group', all(s == 4 for s in sizes), f"group sizes {sizes}"))

    # 4. no duplicate within a group; no team in more than one group
    dup_in_group = any(len(set(v)) != len(v) for v in GROUPS.values())
    across = [t for team_list in GROUPS.values() for t in team_list]
    dup_across = len(across) != len(set(across))
    results.append(('no duplicates', not dup_in_group and not dup_across, "each team appears once"))

    # 5, 6 and 7. validate every fixture
    all_fixture_ok = True
    for f in fixtures:

        # 5. teams belong to the fixture's group
        group_teams = GROUPS[f['group']]
        teams_ok = (f['team_a'] in group_teams and f['team_b'] in group_teams
                    and f['team_a'] != f['team_b'])

        # 6. no missing required fields.
        #    host_nation_if_applicable is None on neutral matches BY DESIGN,
        #    so 'missing' means the field itself is absent, not that it is None.
        missing = [k for k in REQUIRED_FIELDS if k not in f]

        # 7. host / neutral consistency
        if f['host_nation_if_applicable'] is not None:
            host_ok = (not f['neutral']
                       and f['host_nation_if_applicable'] in (f['team_a'], f['team_b'])
                       and (f['venue'].startswith('TBD') is False)
                       and VENUE_COUNTRY.get(f['venue']) == HOST_HOME_COUNTRY.get(
                           dataset_name(f['host_nation_if_applicable'])))
        else:
            host_ok = f['neutral'] is True

        if not (teams_ok and not missing and host_ok):
            all_fixture_ok = False
            results.append((f['match_id'], False, f"teams_ok={teams_ok} missing={missing}"))

    results.append(('all fixtures valid', all_fixture_ok, f"{len(fixtures)} fixtures checked"))
    return results

report = run_validations()
pd.DataFrame(report, columns=['Check', 'Passed', 'Detail'])


,Check,Passed,Detail
0,exactly 48 teams,True,48 teams
1,exactly 12 groups,True,12 groups
2,4 teams per group,True,"group sizes [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]"
3,no duplicates,True,each team appears once
4,all fixtures valid,True,72 fixtures checked


# Section 6 - The knockout bracket (no teams needed yet)

The knockout bracket does **not** need to know which teams advance - it only describes the
**shapes of the rounds**. The advancing teams come out of the (future) simulation of the group
stage.

### What 32 teams qualify?

From 12 groups: every group's **winner**, every group's **runner-up** (24 teams) and the **8 best
third-placed teams**.

We represent the bracket as:

- A list of 32 **qualification slots** (e.g. `Winner A`, `Runner-up L`, `3rd-place #2`).
- Round structures: Round of 32, Round of 16, Quarter-finals, Semi-finals, Third place, Final.
- Each bracket match references two qualification slots, not concrete teams.

### Note on exact pairings

The precise cross-pairings (which group's winner meets which third-place team, etc.) follow
FIFA's official bracket sheet. We show the slot vocabulary and the round skeleton here, and mark
the exact match-to-slot assignments as **pending the official bracket sheet** rather than inventing
them.

In [8]:
# 32 qualification slots available from the group stage.
qualification_slots = (
    [f"Winner of Group {g}" for g in 'ABCDEFGHIJKL']
    + [f"Runner-up of Group {g}" for g in 'ABCDEFGHIJKL']
    + [f"Best 3rd-place team #{k}" for k in range(1, 9)]
)

print("Number of qualification slots:", len(qualification_slots))
qualification_slots[:14]

Number of qualification slots: 32


['Winner of Group A',
 'Winner of Group B',
 'Winner of Group C',
 'Winner of Group D',
 'Winner of Group E',
 'Winner of Group F',
 'Winner of Group G',
 'Winner of Group H',
 'Winner of Group I',
 'Winner of Group J',
 'Winner of Group K',
 'Winner of Group L',
 'Runner-up of Group A',
 'Runner-up of Group B']

In [9]:
# Round skeleton - the competition format, with match counts (no teams).
BRACKET_ROUNDS = [
    {'round': 'Round of 32',       'matches': 16},
    {'round': 'Round of 16',       'matches': 8},
    {'round': 'Quarter-finals',    'matches': 4},
    {'round': 'Semi-finals',       'matches': 2},
    {'round': 'Third-place match', 'matches': 1},
    {'round': 'Final',             'matches': 1},
]
pd.DataFrame(BRACKET_ROUNDS)

,round,matches
0,Round of 32,16
1,Round of 16,8
2,Quarter-finals,4
3,Semi-finals,2
4,Third-place match,1
5,Final,1


In [10]:
def build_knockout_bracket():
    """Represent the bracket as empty match slots that reference qualification
    slots rather than concrete teams. Pairings are pending the official bracket sheet."""
    bracket = {}
    for rd in BRACKET_ROUNDS:
        name = rd['round']
        bracket[name] = [
            {
                'match': f"{name.replace(' ', '_')}_M{m + 1}",
                'team_a_slot': 'PENDING - official bracket sheet',
                'team_b_slot': 'PENDING - official bracket sheet',
                'status': 'pairing pending',
            }
            for m in range(rd['matches'])
        ]
    return bracket

bracket = build_knockout_bracket()

print("Knockout rounds represented:", list(bracket.keys()))
print("Round of 32 slot count:", len(bracket['Round of 32']))
bracket['Round of 32'][:3]

Knockout rounds represented: ['Round of 32', 'Round of 16', 'Quarter-finals', 'Semi-finals', 'Third-place match', 'Final']
Round of 32 slot count: 16


[{'match': 'Round_of_32_M1',
  'team_a_slot': 'PENDING - official bracket sheet',
  'team_b_slot': 'PENDING - official bracket sheet',
  'status': 'pairing pending'},
 {'match': 'Round_of_32_M2',
  'team_a_slot': 'PENDING - official bracket sheet',
  'team_b_slot': 'PENDING - official bracket sheet',
  'status': 'pairing pending'},
 {'match': 'Round_of_32_M3',
  'team_a_slot': 'PENDING - official bracket sheet',
  'team_b_slot': 'PENDING - official bracket sheet',
  'status': 'pairing pending'}]

# Why venue information matters

Venue affects two things in our pipeline:

1. **Home advantage.** A host playing on its own soil is handled by Version 9's `neutral` flag
   (it flips `neutral_encoded` and shifts the predicted probabilities). Getting this right changes
   each host's predicted odds, which can change who advances over many simulations.
2. **Host vs neutral bookkeeping.** By storing `venue`, `city`, `neutral` and
   `host_nation_if_applicable` per fixture, we can later drop in the official schedule and the
   simulator will automatically know which fixtures are true home matches - without us hard-coding
   the host teams' whole schedule.

That is why the venue is part of the tournament structure, not the model.

# Summary

Built a reusable, validated description of the 2026 World Cup:

| Item | Value |
|---|---|
| Teams | 48 (all confirmed qualified teams from the December 5, 2025 final draw) |
| Groups | 12 (A-L), 4 teams each |
| Group-stage matches | 72 (6 per group, generated from rosters) |
| Venues | 16 (11 USA / 3 Mexico / 2 Canada) |
| Knockout | Round of 32 / Round of 16 / QF / SF / 3rd / Final (32 qualify: 12 winners + 12 runners-up + 8 best thirds) |

### How host matches are identified

A fixture's `neutral` is `False` and `host_nation_if_applicable` is set **only** when a host team
plays in a venue in its own country. Right now that is the three confirmed host opening fixtures
(Mexico at Mexico City, USA at Los Angeles, Canada at Toronto); every other fixture is `neutral`
until its venue is filled in. Version 9 uses this to flip `neutral_encoded`, e.g. `USA` plays as
`neutral=False` only in the USA opener.

### How the knockout bracket is represented

As a qualification-slot vocabulary (32 slots) plus round skeletons whose matches reference slots
instead of concrete teams - so it works without knowing who advances.

### Still missing before we can build the simulator

1. **Official per-match venues and dates** for the full group stage (currently `TBD`) so host
   matches beyond the three openers are correctly flagged.
2. **The official bracket sheet's exact pairings** (e.g. which third-place team meets which
   group winner / runner-up) to fill the bracket match slots.
3. Confirming how the **8 best third-placed teams** are ranked (points, goal difference, etc.) -
   the tie-break rule we will implement in the simulator.
4. Then the simulator itself: play the group stage, apply these advancement rules, sample every
   match from Version 9's predictions, and run many tournaments.